In [4]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report , accuracy_score
import numpy as np



In [5]:
df = pd.read_csv(r"E:\fastapi\insurance (1).csv")


In [6]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
59,61,118.7,1.82,1.13,False,Jaipur,retired,High
61,32,102.4,1.68,24.05,True,Chandigarh,unemployed,High
65,46,106.3,1.68,38.07,True,Jaipur,unemployed,High
23,35,70.3,1.78,23.71,False,Mysore,unemployed,Medium
56,24,101.9,1.55,2.86,True,Kolkata,student,Medium


In [35]:
df['occupation'].unique()

array(['retired', 'freelancer', 'student', 'government_job',
       'business_owner', 'unemployed', 'private_job'], dtype=object)

In [7]:
df_feat = df.copy()

In [8]:
df_feat["bmi"] = df_feat["weight"] / (df_feat['height']**2)

In [9]:
def age_group(age):
    if age < 25 :
        return "young"
    elif age < 45:
        return "adult"
    elif age < 60 :
        return "middle_aged"
    return "senior"

In [10]:
df_feat["age_group"] = df_feat['age'].apply(age_group)

In [11]:
def lifestyle_risk(row):
    if row['smoker'] and row["bmi"] > 30:
        return "high"
    elif row["smoker"] or row["bmi"] > 27:
        return "medium"
    else:
        return "low"

In [12]:
df_feat['lifestyle_risk'] = df_feat.apply(lifestyle_risk , axis = 1)

In [14]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

In [15]:
def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    else:
        return 3

In [16]:
df_feat["city_tier"] = df_feat["city"].apply(city_tier)

In [25]:
df_feat.drop(columns=['age','weight','height','smoker','city'])[['income_lpa','occupation','bmi','age_group','lifestyle_risk','city_tier','insurance_premium_category']].sample(5)

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
31,11.77,private_job,15.258742,adult,medium,2,Medium
65,38.07,unemployed,37.662982,middle_aged,high,2,High
4,3.94,retired,24.296875,senior,medium,2,High
86,37.38,freelancer,18.476526,adult,low,1,Low
12,17.58,freelancer,30.046711,adult,high,2,High


In [31]:
X = df_feat[["bmi","age_group","lifestyle_risk","city_tier","income_lpa","occupation"]]
y = df_feat["insurance_premium_category"]

In [27]:
categorical_features = ["age_group","lifestyle_risk","occupation","city_tier"]
numeric_features = ["bmi", "income_lpa"]


In [28]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), categorical_features),
        ("num","passthrough", numeric_features)
    ]
)


In [29]:
pipeline = Pipeline(steps=[
    ("preprocessor",preprocessor),
    ("classifier",RandomForestClassifier(random_state = 42))
])

In [32]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size = 0.2 , random_state = 1)
pipeline.fit(X_train , y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'occupation', 'city_tier']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_lpa'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [33]:
y_pred = pipeline.predict(X_test)
accuracy_score(y_test , y_pred)

0.9

In [34]:
import pickle
pickle_model_path = "model.pkl"
with open(pickle_model_path,"wb") as f:
    pickle.dump(pipeline,f)